# Measuring What Matters: Benchmarking and Evaluation

## Setup

So here again in this environment, we've already pre-warmed the vLLM server with the same model Qwen3 .6B. Let's send a quick test request just to confirm everything is working.

In [4]:
import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, glob
from openai import OpenAI

VLLM_URL = "http://localhost:8000"

for _ in range(12):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        time.sleep(5)
else:
    raise RuntimeError("vLLM server not reachable.")
    
print(f"Connected to {VLLM_URL} — model: {MODEL}")

Connected to http://localhost:8000 — model: Qwen/Qwen3-0.6B


And let's send this request "What is model quantization in one sentence"

In [3]:
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")
resp = client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", 
               "content": "What is model quantization in one sentence."}],
    max_tokens = 30, 
    temperature=0.7,
    extra_body ={"chat_template_kwargs": {"enable_thinking": False}},
)
print(f"{MODEL}: {resp.choices[0].message.content.strip()}")

Qwen/Qwen3-0.6B: Model quantization is a technique used to reduce the size and computational complexity of deep neural networks by mapping high-dimensional input and output data to a smaller set


Okay great, all looks good! Let's use GuideLLM to benchmark the performance of our model. 

## Benchmarking with GuideLLM

First we'll create the directory `outputs` where we'll save the benchmark results.

In [6]:
os.makedirs("outputs", exist_ok=True)

Here's the command you can use to run guidellm. You can run it from the terminal but here we're running from this code cell using the `subprocess` module. 

Let's break down the flags we're passing:

- target http://localhost:8000 — points GuideLLM at our local vLLM server. It hits the OpenAI-compatible endpoints
- profile synchronous — sends requests one at a time, waiting for each to finish before sending the next. This gives us a clean baseline of single-request latency, with no batching or queueing in the picture. Other profiles (concurrent, throughput, sweep) ramp up load to stress-test how the server holds up under traffic.
- max-requests 10 — caps the run at 10 requests. This is deliberately tiny so the cell finishes quickly in this environment; a real benchmark would use a few hundred to a few thousand requests, or switch to a time-bounded run with --max-seconds.
- data prompt_tokens=32,output_tokens=16,samples=32 — synthetic workload spec: each request is ~32 prompt tokens in, ~16 output tokens out, drawn from a pool of 32 pre-generated prompts. Keeping samples >= max-requests means no prompt repeats, so we don't accidentally inflate prefix-cache hits.
- output-dir ./outputs — where GuideLLM writes the JSON report
  
**Run the cell now, then say**It'll take a moment to finish. When it's done, you'll have a structured benchmark file with per-request distributions, ready to interpret. 

**Make sure to show the results (the table below), explain what the numbers mean and mention that these results are stored in two formats: json and csv.** Now, GuideLLM saves these results in two formats: JSON and CSV with pre-computed statistics for means, percentiles, min and max, so you don't have to calculate it outself.

In [7]:
import subprocess

cmd = [
    "guidellm", "benchmark",
    "--target", "http://localhost:8000",
    "--model", MODEL,
    "--profile", "synchronous",
    "--max-requests", "10",
    "--data", "prompt_tokens=32,output_tokens=16,samples=32",
    "--output-dir", "./outputs",
]

print(f"Running: {' '.join(cmd)}\n")

result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

if result.returncode == 0:
    print("Benchmark complete!")
    tail = result.stdout[-2000:] if len(
        result.stdout) > 2000 else result.stdout
    print(tail)
else:
    print(f"GuideLLM exited with code {result.returncode}")
    print(f"STDOUT:\n{result.stdout[-1000:]}")
    print(f"STDERR:\n{result.stderr[-1000:]}")

Running: guidellm benchmark --target http://localhost:8000 --model Qwen/Qwen3-0.6B --profile synchronous --max-requests 10 --data prompt_tokens=32,output_tokens=16,samples=32 --output-dir ./outputs

Benchmark complete!

| synchronous | 1.7     | 2.0    | 122.2 | 292.5 | 101.7 | 123.2 | 105.6 | 125.5 |
|=============|=========|========|=======|=======|=======|=======|=======|=======|


ℹ Server Throughput Statistics (All Requests)
|=============|=======|======|=========|==============|===============|==============|
| Benchmark   | Requests             ||| Input Tokens | Output Tokens | Total Tokens |
| Strategy    | Concurrency || Per Sec | Per Sec      | Per Sec       | Per Sec      |
|             | Mdn   | Mean | Mean                                               ||||
|-------------|-------|------|---------|--------------|---------------|--------------|
| synchronous | 1.0   | 1.0  | 0.6     | 26.9         | 9.8           | 34.4         |
|=============|=======|======|=========|====

## Interpreting Benchmark Results

LEt's now extract from the json files the key numbers: 
- the Time to first token
- the inter token latence
- the end to end latency 

**Run the cell**. So here's the mean, the 50th, 95th and 99th percentiles of each metric. This is what you'd put in a report to your team to use or evaluate whether an LLM deployment meets your SLO's.

Since averages hide outliers, you might notice the p95, meaning 95% of requests are faster than that value, is worse, and same with the p99. That's really important because 5 in 100 users could be waiting 20 times longer than usual.

When evaluating a deployment, always look at the p95 and p99, because if there's a big gap between the mean and the p95, you've got tail latency issues, and users will be sure to feel those issues.

In [8]:
with open("outputs/benchmarks.json") as f:
    report = json.load(f)

bench = report["benchmarks"][0]
metrics = bench["metrics"]
n_requests = metrics["request_totals"]["successful"]

print(f"Profile: {bench['type_']}  |  Requests: {n_requests}\n")

display_metrics = {
    "TTFT (ms)":       "time_to_first_token_ms",
    "ITL (ms)":        "inter_token_latency_ms",
    "E2E Latency (s)": "request_latency",
    "Output tokens":   "output_token_count",
}

print(f"{'Metric':<20} {'Mean':>8} {'p50':>8} {'p95':>8} {'p99':>8}")
print("-" * 55)
for label, key in display_metrics.items():
    dist = metrics[key]["successful"]
    p = dist["percentiles"]
    print(f"{label:<20} {dist['mean']:>8.2f} {p['p50']:>8.2f} "
          f"{p['p95']:>8.2f} {p['p99']:>8.2f}")

throughput = metrics["output_tokens_per_second"]["successful"]
req_rate = metrics["requests_per_second"]["successful"]
print(f"\nThroughput: {req_rate['mean']:.2f} req/s  |  "
      f"{throughput['mean']:.1f} output tokens/s")

Profile: generative_benchmark  |  Requests: 10

Metric                   Mean      p50      p95      p99
-------------------------------------------------------
TTFT (ms)              141.10   122.24   292.45   292.45
ITL (ms)                99.98   101.74   123.18   123.18
E2E Latency (s)          1.64     1.69     2.01     2.01
Output tokens           16.00    16.00    16.00    16.00

Throughput: 0.61 req/s  |  9.8 output tokens/s


Now, performance benchmarks tell you how fast a deployment is, but not whether the model gives good answers. That's a totally different type of test. You could have the fastest inference server in the world, but if the model's accuracy drops 15% from quantization, it's not deployable. That's where lm_eval comes in.

## Evaluating Model Quality with lm_eval

lm_eval is a standardized evaluation harness that measures task performance, or how well a model answers on knowledge, reasoning, and coding benchmarks. While GuideLLM asks "how well does this deployment perform", lm_eval will help us determine "how well this model answers".

We'll point lm_eval at the same running server using the OpenAI completions endpoint, just like earlier, and run Hellaswag, a common sense reasoning benchmark. It's the same evaluation that appears on most model cards.

We're using lm_evals simple evaluate function on 20 examples to keep it quick, but in production, you'd run the full test set once or multiple times. 

**[mention what other tasks we could give to lm_eval, can the define their own task?]**

Let's run the cell. **[Run the cell]** It'll take a moment to finish. **wait for cell to be done** Now that the results are done, let's look at them. 


In [9]:
import lm_eval

os.environ.setdefault("OPENAI_API_KEY", "unused")

TASK = "hellaswag"
print(f"Running lm_eval on {MODEL} via vLLM server ({TASK}, 20 examples)...\n")

results = lm_eval.simple_evaluate(
    model="local-completions",
    model_args=(
        f"model={MODEL},"
        f"base_url={VLLM_URL}/v1/completions,"
        "tokenized_requests=False,"
        "num_concurrent=1"
    ),
    tasks=[TASK],
    limit=20,
)

Running lm_eval on Qwen/Qwen3-0.6B via vLLM server (hellaswag, 20 examples)...



Requesting API: 100%|██████████| 80/80 [00:12<00:00,  6.61it/s]


We'll print out the accuracy metrics, which it looks like we've got a 30% with a standard deviation of around 3%, which isn't the best, but remember this is a model that could probably fit on a smart fridge. 

In [10]:
task_results = results["results"][TASK]

print(f"Model: {MODEL}")
print(f"Task: {TASK}  |  Examples: 20\n")
for metric, value in task_results.items():
    if isinstance(value, (int, float)):
        print(f"  {metric}: {value:.4f}")


Model: Qwen/Qwen3-0.6B
Task: hellaswag  |  Examples: 20

  acc,none: 0.3000
  acc_stderr,none: 0.1051
  acc_norm,none: 0.3000
  acc_norm_stderr,none: 0.1051


These evaluations give you numerical evidence to make educated model deployment decisions, from both deployment performance using GuideLLM, to model accuracy using lm_eval. You can also use published model card data that's been evaluated by the publisher, before you even get started.

## Reading Published Benchmark Data

In a previous lesson, you learned how to quantize a model with `llm-compressor` using GPTQ. In practice, quantized model publishers include **accuracy tables** on their model cards so users can evaluate the tradeoff without running every benchmark themselves.

Here's the accuracy table from the [RedHatAI/Qwen3-0.6B-quantized.w4a16 model card](https://huggingface.co/RedHatAI/Qwen3-0.6B-quantized.w4a16) — the W4A16 variant of the model we've been working with. The **recovery** column shows how much of the base model's accuracy the quantized version retains. Most benchmarks with meaningful base scores show 97–100% recovery, a strong result for a major model size reduction.

Also note that the accuracy of the model using Hellaswag is 43% which is higher than the 30% we obtained. That's because we only used 20 samples and a single shot. 

| Category | Benchmark | Qwen3-0.6B | W4A16 (this model) | Recovery |
|:--|:--|--:|--:|--:|
| **OpenLLM v1** | MMLU (5-shot) | 42.82 | 39.80 | 93.0% |
| | ARC Challenge (25-shot) | 32.85 | 30.72 | 93.5% |
| | GSM-8K (5-shot) | 1.82 | 2.20 | — |
| | Hellaswag (10-shot) | 43.04 | 41.02 | 95.3% |
| | Winogrande (5-shot) | 54.54 | 54.62 | 100.1% |
| | TruthfulQA (0-shot) | 51.61 | 48.77 | 94.5% |
| | **Average** | **37.78** | **36.19** | **95.8%** |
| **OpenLLM v2** | MMLU-Pro (5-shot) | 17.25 | 14.27 | — |
| | IFEval (0-shot) | 62.83 | 55.81 | 88.8% |
| | BBH (3-shot) | 4.23 | 1.63 | — |
| | Math-lvl-5 (4-shot) | 18.26 | 10.26 | — |
| | GPQA (0-shot) | 0.00 | 0.00 | — |
| | MuSR (0-shot) | 0.00 | 0.00 | — |
| | **Average** | **17.10** | **13.66** | — |
| **Multilingual** | MGSM (0-shot) | 19.70 | 19.90 | — |
| **Reasoning** | AIME 2024 | 9.69 | 3.44 | — |
| | AIME 2025 | 13.13 | 6.98 | — |
| | GPQA diamond | 29.29 | 27.78 | 94.8% |
| | Math-lvl-5 | 71.60 | 70.60 | 98.6% |
| | LiveCodeBench | 12.83 | 8.35 | — |


You now have three sources of evidence:

| Source | What it tells you |
|:--|:--|
| **GuideLLM** | How the deployment *performs* (latency, throughput, consistency) |
| **lm_eval** | How the model *answers* (accuracy on a task you ran yourself) |
| **Model card** | How the model performs across many benchmarks, evaluated by the publisher |

When deciding whether to deploy a quantized model, you need **both** dimensions. An optimization that doubles throughput but drops accuracy by 15% may not be worth it. An accurate model that can't meet latency SLOs isn't deployable either.

For this W4A16 model: **~50% model size reduction** for **~4% average accuracy loss** on OpenLLM v1. The answer depends on your use case — check recovery on the tasks that matter to you.

Now join me in the next and final lesson, where we'll put together all the concepts you learned about in this course.